# Data Explorer: PubSub OHLCV

Analyze PubSub crawl data with OHLCV candlestick charts.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import mplfinance as mpf  # Uncomment if installed

## Configuration

In [2]:
# Config
PUBSUB_DIR = "./data/pubsub"

# Stock to analyze
STOCK = "41I1G3000"

# Time interval for OHLCV (e.g., '1min', '5min', '15min', '1h')
TIME_INTERVAL = '1min'

## Load Data

In [3]:
# Find latest date with data
def get_latest_date(data_dir, stock):
    stock_dir = os.path.join(data_dir, stock)
    if not os.path.exists(stock_dir):
        return None
    files = glob.glob(os.path.join(stock_dir, "*.fea"))
    if not files:
        return None
    dates = [os.path.splitext(os.path.basename(f))[0] for f in files]
    return max(dates)

latest_date = get_latest_date(PUBSUB_DIR, STOCK)
print(f"Latest date: {latest_date}")

Latest date: 2026-03-13


In [4]:
# Load PubSub data
if latest_date:
    pubsub_path = os.path.join(PUBSUB_DIR, STOCK, f"{latest_date}.fea")
    df = pd.read_feather(pubsub_path)
    print(f"Loaded {len(df)} rows from {pubsub_path}")
else:
    df = None
    print("No data found")

Loaded 221964 rows from ./data/pubsub/41I1G3000/2026-03-13.fea


In [5]:

df.tail(120)

,timestamp,symbol,match_price,match_volume,total_volume,high_price,low_price,price_change,price_change_percent,ceiling_price,...,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,foreign_buy_volume,foreign_sell_volume,foreign_room,msg_id
221844,2026-03-13T07:45:00.558000,41I1G3000,1.84,2,323666,1.8775,1.8345,-0.013,-0.7,1.9827,...,1.8415,1,1.8419,25,1.842,59,11540,14371,0,b'1773387903949-2'
221845,2026-03-13T07:45:00.558000,41I1G3000,1.84,3,323669,1.8775,1.8345,-0.013,-0.7,1.9827,...,1.8415,1,1.8419,25,1.842,59,11540,14371,0,b'1773387903950-0'
221846,2026-03-13T07:45:00.561000,41I1G3000,1.84,3,323672,1.8775,1.8345,-0.013,-0.7,1.9827,...,1.8415,1,1.8419,25,1.842,59,11540,14371,0,b'1773387903950-1'
221847,2026-03-13T07:45:00.561000,41I1G3000,1.84,4,323677,1.8775,1.8345,-0.013,-0.7,1.9827,...,1.8415,1,1.8419,25,1.842,59,11540,14371,0,b'1773387903950-2'
221848,2026-03-13T07:45:00.561000,41I1G3000,1.84,3,323680,1.8775,1.8345,-0.013,-0.7,1.9827,...,1.8415,1,1.8419,25,1.842,59,11540,14371,0,b'1773387903951-0'
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221959,2026-03-13T08:02:00.088000,41I1G3000,1.84,100,323969,1.8775,1.8345,-0.013,-0.7,1.9827,...,1.8415,1,1.8419,25,1.842,59,11619,15607,0,b'1773388920529-0'
221960,2026-03-13T08:02:30.088000,41I1G3000,1.84,100,323969,1.8775,1.8345,-0.013,-0.7,1.9827,...,1.8415,1,1.8419,25,1.842,59,11619,15607,0,b'1773388950367-0'
221961,2026-03-13T08:03:00.088000,41I1G3000,1.84,100,323969,1.8775,1.8345,-0.013,-0.7,1.9827,...,1.8415,1,1.8419,25,1.842,59,11619,15607,0,b'1773388980228-0'
221962,2026-03-13T08:03:30.088000,41I1G3000,1.84,100,323969,1.8775,1.8345,-0.013,-0.7,1.9827,...,1.8415,1,1.8419,25,1.842,59,11619,15607,0,b'1773389010401-0'


## OHLCV Converter

In [6]:
import pandas as pd
import datetime

def tick_to_ohlcv(df, interval='1min', price_col='match_price', volume_col='match_volume'):
    """
    Convert tick data to OHLCV candlestick data, filtering valid market hours.
    """
    if df is None or df.empty:
        return None
    
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], format="ISO8601") + pd.Timedelta(hours=7)
    tick_times = df['timestamp'].dt.time
    
    # Define valid market windows
    morning_valid = tick_times <= datetime.time(11, 30, 0)
    afternoon_valid = (tick_times >= datetime.time(13, 0, 0)) & (tick_times <= datetime.time(14, 30, 0))
    
    # Keep only rows that fall in the morning OR afternoon session
    df = df[morning_valid | afternoon_valid]
    # ---------------------------------------------------------
    
    df = df.set_index('timestamp')
    
    # pandas .ohlc() automatically creates 'open', 'high', 'low', 'close'
    ohlcv = df[price_col].resample(interval).ohlc()
    ohlcv['volume'] = df[volume_col].resample(interval).sum()
    
    # Drop periods with no trading activity
    ohlcv = ohlcv.dropna(subset=['open']).reset_index()
    
    return ohlcv

In [7]:
ohlcv = tick_to_ohlcv(df, interval=TIME_INTERVAL)

ohlcv.head()

,timestamp,open,high,low,close,volume
0,2026-03-13 13:04:00,1.8629,1.8630,1.8615,1.8623,3851
1,2026-03-13 13:05:00,1.8623,1.8627,1.8618,1.8621,2702
2,2026-03-13 13:06:00,1.8621,1.8632,1.8618,1.8625,4416
3,2026-03-13 13:07:00,1.8625,1.8630,1.8617,1.8618,5343
4,2026-03-13 13:08:00,1.8618,1.8620,1.8610,1.8618,4510


## Visualize OHLCV

In [8]:

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Convert to OHLCV
if df is not None:
    ohlcv = tick_to_ohlcv(df, interval=TIME_INTERVAL)
    print(f"OHLCV - {TIME_INTERVAL}: {len(ohlcv)} candles")
    
    # Determine Volume bar colors (Green for bullish, Red for bearish)
    volume_colors = ['#26a69a' if row['close'] >= row['open'] else '#ef5350' for _, row in ohlcv.iterrows()]

    # 2. Create interactive subplots (Row 1 = Candlestick, Row 2 = Volume)
    fig = make_subplots(
        rows=2, cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.03, 
        subplot_titles=(f'{STOCK} Price', 'Volume'), 
        row_width=[0.2, 0.7]  # 70% height for price, 20% for volume
    )

    # 3. Add Candlestick Trace
    fig.add_trace(
        go.Candlestick(
            x=ohlcv['timestamp'],
            open=ohlcv['open'],
            high=ohlcv['high'],
            low=ohlcv['low'],
            close=ohlcv['close'],
            name='Price',
            increasing_line_color='#26a69a', # Custom green
            decreasing_line_color='#ef5350'  # Custom red
        ),
        row=1, col=1
    )

    # 4. Add Volume Trace
    fig.add_trace(
        go.Bar(
            x=ohlcv['timestamp'],
            y=ohlcv['volume'], 
            name='Volume',
            marker_color=volume_colors
        ),
        row=2, col=1
    )

    # 5. Format layout
    fig.update_layout(
        title=f"{STOCK} - {TIME_INTERVAL} Interactive OHLCV Chart",
        xaxis_rangeslider_visible=False, # Hides the redundant rangeslider at the bottom
        height=700,
        template='plotly_white',
        showlegend=False
    )
    
    # Tweak y-axis to not start at 0 for price
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Volume", row=2, col=1)

    # Add this right before fig.show() to hide the lunch gap!
    fig.update_xaxes(
        rangebreaks=[
            dict(bounds=["11:30", "13:00"], pattern="hour"), # Hides the lunch break
            dict(bounds=["14:30", "09:00"], pattern="hour")  # Hides overnight gap (if plotting multiple days)
        ]
    )

    fig.show()
else:
    print("No OHLCV data to plot")

OHLCV - 1min: 86 candles
